# SPADE Benchmark: Ensemble vs Rigid Docking
Compares classical rigid docking against SPADE ensemble docking across two targets and a mixed ligand set.

In [ ]:
import subprocess, sys, shutil

# Core dependencies
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "rdkit", "prody", "prolif", "numba", "meeko", "gemmi",
    "pandas", "plotly"], check=False)
# Install SPADE from GitHub
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "git+https://github.com/ChandraguptSharma07/Spade.git"], check=False)

# --- UniDock binary: try CUDA 12.2 first, fall back to 12.0 ---
_UNIDOCK_PATH = "/usr/local/bin/unidock"
_UNIDOCK_RELEASES = [
    "https://github.com/dptech-corp/Uni-Dock/releases/download/1.1.0/unidock-1.1.0-cuda122-linux-x86_64",
    "https://github.com/dptech-corp/Uni-Dock/releases/download/1.1.0/unidock-1.1.0-cuda120-linux-x86_64",
]

def _install_unidock():
    if shutil.which("unidock"):
        return True   # already on PATH (e.g. conda env)
    for url in _UNIDOCK_RELEASES:
        r = subprocess.run(["wget", "-q", url, "-O", _UNIDOCK_PATH], capture_output=True)
        if r.returncode != 0:
            continue
        subprocess.run(["chmod", "+x", _UNIDOCK_PATH], check=True)
        # Smoke-test: UniDock exits 1 on --help but still prints, which is fine
        probe = subprocess.run([_UNIDOCK_PATH, "--help"], capture_output=True, timeout=10)
        if b"UniDock" in probe.stdout + probe.stderr or probe.returncode in (0, 1):
            print(f"UniDock installed from: {url.split('/')[-1]}")
            return True
        # Bad binary — remove and try next
        import os; os.unlink(_UNIDOCK_PATH)
    return False

_unidock_available = _install_unidock()
if not _unidock_available:
    print("WARNING: UniDock could not be installed. Will use CPU backend.")

# --- CUDA diagnostics ---
def _cuda_info():
    try:
        r = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=10)
        if r.returncode == 0:
            for line in r.stdout.splitlines():
                if "CUDA Version" in line or "Driver Version" in line:
                    print(line.strip())
            gpus = subprocess.run(
                ["nvidia-smi", "--query-gpu=index,name,memory.total,utilization.gpu",
                 "--format=csv,noheader"],
                capture_output=True, text=True, timeout=10
            )
            for g in gpus.stdout.strip().splitlines():
                print(f"  GPU {g}")
            return True
    except Exception as e:
        print(f"nvidia-smi error: {e}")
    return False

_gpu_present = _cuda_info()

# --- Backend selection ---
def _detect_backend():
    if _gpu_present and _unidock_available:
        print("Backend: gpu (UniDock --gpu_batch)")
        return "gpu"
    if _gpu_present:
        print("GPU detected but UniDock unavailable — falling back to CPU (Vina)")
    else:
        print("No GPU detected — using CPU (Vina)")
    return "cpu"

BACKEND = _detect_backend()
print(f"\nBACKEND = {BACKEND!r}")

In [ ]:
## UniDock smoke-test — run this before docking to confirm GPU is actually used
import subprocess, tempfile, os, textwrap

_TINY_RECEPTOR = textwrap.dedent("""\
    ATOM      1  CA  ALA A   1       0.000   0.000   0.000  1.00  0.00     0.000 C
    TER
""")
_TINY_LIGAND = textwrap.dedent("""\
    REMARK  Name = test
    ROOT
    ATOM      1  C1  LIG L   1       0.100   0.100   0.100  1.00  0.00     0.000 C
    ENDROOT
    TORSDOF 0
""")

with tempfile.TemporaryDirectory() as td:
    rec = os.path.join(td, "rec.pdbqt")
    lig = os.path.join(td, "lig.pdbqt")
    out = os.path.join(td, "out.pdbqt")
    with open(rec, "w") as f: f.write(_TINY_RECEPTOR)
    with open(lig, "w") as f: f.write(_TINY_LIGAND)

    cmd = [
        "unidock",
        "--receptor", rec,
        "--ligand", lig,
        "--out", out,
        "--center_x", "0", "--center_y", "0", "--center_z", "0",
        "--size_x", "20", "--size_y", "20", "--size_z", "20",
        "--num_modes", "1", "--exhaustiveness", "4",
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
    print("=== UniDock stdout ===")
    print(result.stdout[:3000] or "(empty)")
    print("=== UniDock stderr ===")
    print(result.stderr[:3000] or "(empty)")
    print("=== exit code:", result.returncode, "===")

## Targets
- **EGFR** (P00533) — well-folded kinase, clinical benchmark  
- **CDK2** (P24941) — flexible glycine-rich loop, shows ensemble benefit

In [ ]:
from spade.core.structure import fetch_structure
import numpy as np
import time

print("Fetching EGFR (P00533)...")
egfr = fetch_structure("P00533")

print("Fetching CDK2 (P24941)...")
cdk2 = fetch_structure("P24941")

TARGETS = {
    "EGFR": {
        "structure": egfr,
        "pocket_residues": np.arange(695, 720),   # ATP hinge, 0-indexed
    },
    "CDK2": {
        "structure": cdk2,
        "pocket_residues": np.array(list(range(9, 20)) + list(range(79, 90)) + list(range(128, 150))),
    },
}
print("Targets ready.")

## Ligand Set
| Compound | Category |
|---|---|
| Erlotinib | EGFR binder |
| Gefitinib | EGFR binder |
| Flavopiridol | CDK2 binder |
| Roscovitine | CDK2 binder |
| Imatinib | Off-target kinase |
| Curcumin | Aggregator (decoy) |
| Ibuprofen | Non-binder (decoy) |
| Metformin | Non-binder (decoy) |
| Aspirin | Non-binder (decoy) |
| Caffeine | Non-binder (decoy) |

In [ ]:
from spade.core.ligand import prepare_ligand

COMPOUNDS = {
    "Erlotinib":    {"smiles": "CCOc1cc2ncnc(Nc3cccc(C#C)c3)c2cc1OCCOCCO",                        "category": "EGFR binder"},
    "Gefitinib":    {"smiles": "COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1",                 "category": "EGFR binder"},
    "Flavopiridol": {"smiles": "OC1=CC2=C(C=C1)N(C1CCN(C)CC1)C(=O)C=C2Cl",                       "category": "CDK2 binder"},
    "Roscovitine":  {"smiles": "CCC(CO)(Cc1cccnc1)NC1=NC(=NC2=CN=CN12)NC(C)C",                   "category": "CDK2 binder"},
    "Imatinib":     {"smiles": "Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc(-c2cccnc2)n1",    "category": "Off-target"},
    "Curcumin":     {"smiles": "COc1cc(/C=C/C(=O)CC(=O)/C=C/c2ccc(O)c(OC)c2)ccc1O",             "category": "Aggregator"},
    "Ibuprofen":    {"smiles": "CC(C)Cc1ccc(cc1)C(C)C(=O)O",                                     "category": "Decoy"},
    "Metformin":    {"smiles": "CN(C)C(=N)NC(N)=N",                                              "category": "Decoy"},
    "Aspirin":      {"smiles": "CC(=O)Oc1ccccc1C(=O)O",                                          "category": "Decoy"},
    "Caffeine":     {"smiles": "Cn1c(=O)c2c(ncn2C)n(C)c1=O",                                     "category": "Decoy"},
}

ligand_vault = {}
failed = []
for name, data in COMPOUNDS.items():
    variants = prepare_ligand(data["smiles"], ph=7.4, enumerate_stereo=False)
    if variants:
        ligand_vault[name] = variants[0]
        print(f"  {name}: OK")
    else:
        failed.append(name)
        print(f"  {name}: FAILED — will be skipped")

print(f"\n{len(ligand_vault)}/10 ligands prepared. Failed: {failed}")

## Phase 1: Classical Rigid Docking
Single reference structure. One call per ligand per target.

In [ ]:
from spade.core.docking import compute_bounding_box, get_docking_engine
import warnings, time

rigid_results = {}
rigid_times  = {}

ligand_list  = list(ligand_vault.values())
ligand_names = list(ligand_vault.keys())

# GPU: search_mode="fast" (exhaustiveness=128, max_step=20) fills T4 CUDA cores
# CPU: exhaustiveness=8 is standard Vina default
GPU_SEARCH_MODE = "fast"

for tname, tdata in TARGETS.items():
    rigid_results[tname] = {}
    ref_conformer = tdata["structure"].atoms
    bbox = compute_bounding_box(ref_conformer, tdata["pocket_residues"])
    engine = get_docking_engine(
        backend=BACKEND,
        exhaustiveness=8,
        scoring="vina",
        search_mode=GPU_SEARCH_MODE if BACKEND == "gpu" else None,
    )

    t0 = time.perf_counter()

    if BACKEND == "gpu":
        print(f"[{tname}] batching all {len(ligand_list)} ligands (search_mode={GPU_SEARCH_MODE!r})...")
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            batch_poses = engine.dock_batch(ref_conformer, ligand_list, bbox, n_poses=9, conf_idx=0)
        if caught:
            for w in caught:
                print(f"  WARNING: {w.message}")
        for name, poses in zip(ligand_names, batch_poses):
            score = poses[0].score_kcal_mol if poses else 0.0
            rigid_results[tname][name] = score
            print(f"  [{tname}] {name}: {score:.3f} kcal/mol")
    else:
        for name, lig in zip(ligand_names, ligand_list):
            poses = engine.dock(ref_conformer, lig, bbox, n_poses=9, conf_idx=0)
            score = poses[0].score_kcal_mol if poses else 0.0
            rigid_results[tname][name] = score
            print(f"  [{tname}] {name}: {score:.3f} kcal/mol")

    rigid_times[tname] = time.perf_counter() - t0
    print(f"{tname} rigid docking done in {rigid_times[tname]:.1f}s\n")

## Phase 2: SPADE Ensemble Docking
5-conformer ensemble per target (ANM + PAE-weighted modes).
All ligands batched per conformer via UniDock `--gpu_batch` on GPU.

In [ ]:
from spade.core.flexibility import build_flexibility_profile
from spade.core.ensemble import EnsembleGenerator
from spade.core.docking import dock_ensemble
from spade.core.clustering import cluster_poses

ensemble_results = {}
ensemble_times   = {}

ligand_list  = list(ligand_vault.values())
ligand_names = list(ligand_vault.keys())

N_CONFORMERS = 5   # 5 captures the dominant conformational diversity; use 10 for publication

for tname, tdata in TARGETS.items():
    print(f"\n=== {tname}: Generating ensemble ({N_CONFORMERS} conformers) ===")
    structure = tdata["structure"]
    pocket_residues = tdata["pocket_residues"]
    ca_coords = structure.atoms.select("name CA").getCoords()

    profile = build_flexibility_profile(
        structure.plddt, structure.pae_matrix, pocket_residues, ca_coords, cutoff_angstrom=12.0
    )
    conformers = EnsembleGenerator(structure, profile, n_conformers=N_CONFORMERS).generate()
    print(f"  {len(conformers)} conformers generated")

    ensemble_results[tname] = {}
    t0 = time.perf_counter()

    all_dock_out = dock_ensemble(
        conformers, ligand_list, pocket_residues,
        exhaustiveness=8, n_poses=9,
        backend=BACKEND, scoring="vina",
        search_mode=GPU_SEARCH_MODE if BACKEND == "gpu" else None,
    )

    n_conf = len(conformers)
    n_lig  = len(ligand_list)
    for lig_idx, name in enumerate(ligand_names):
        lig_results = [all_dock_out[c * n_lig + lig_idx] for c in range(n_conf)]
        try:
            consensus = cluster_poses(lig_results, conformers, ligand_vault[name].mol)
            tc = consensus.top_cluster
            ensemble_results[tname][name] = {
                "consensus_score": tc.consensus_score if tc else 0.0,
                "coverage":        tc.fraction_ensemble if tc else 0.0,
                "n_clusters":      consensus.n_clusters,
            }
        except Exception as e:
            print(f"  Clustering failed for {name}: {e}")
            ensemble_results[tname][name] = {"consensus_score": 0.0, "coverage": 0.0, "n_clusters": 0}
        print(f"  [{tname}] {name}: consensus={ensemble_results[tname][name]['consensus_score']:.3f}  coverage={ensemble_results[tname][name]['coverage']:.0%}  clusters={ensemble_results[tname][name]['n_clusters']}")

    ensemble_times[tname] = time.perf_counter() - t0
    print(f"{tname} ensemble done in {ensemble_times[tname]:.1f}s")

## Results

In [ ]:
import pandas as pd

rows = []
for tname in TARGETS:
    for name in ligand_vault:
        rows.append({
            "Target":           tname,
            "Ligand":           name,
            "Category":         COMPOUNDS[name]["category"],
            "Rigid Score":      round(rigid_results[tname].get(name, 0.0), 3),
            "Consensus Score":  round(ensemble_results[tname][name]["consensus_score"], 3),
            "Coverage":         round(ensemble_results[tname][name]["coverage"], 2),
            "Clusters":         ensemble_results[tname][name]["n_clusters"],
            "Rigid Time (s)":   round(rigid_times[tname] / len(ligand_vault), 1),
            "Ensemble Time (s)":round(ensemble_times[tname] / len(ligand_vault), 1),
        })

df = pd.DataFrame(rows)
df.to_csv("spade_benchmark_results.csv", index=False)

# Style: highlight best scores per target
display(df.style
    .background_gradient(subset=["Rigid Score", "Consensus Score"], cmap="RdYlGn_r")
    .background_gradient(subset=["Coverage"], cmap="RdYlGn")
    .format({"Coverage": "{:.0%}"})
)

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

for tname in TARGETS:
    sub = df[df["Target"] == tname].copy()

    fig = px.scatter(
        sub,
        x="Rigid Score",
        y="Coverage",
        color="Category",
        size="Clusters",
        size_max=30,
        text="Ligand",
        title=f"{tname} — Rigid Score vs Ensemble Coverage<br><sup>Binders: top-right | Decoys: bottom-left</sup>",
        labels={"Rigid Score": "Rigid Docking Score (kcal/mol)", "Coverage": "Ensemble Coverage"},
        color_discrete_map={
            "EGFR binder": "#00cc66",
            "CDK2 binder": "#3399ff",
            "Off-target":  "#ff9900",
            "Aggregator":  "#cc00cc",
            "Decoy":       "#888888",
        },
    )
    fig.update_traces(textposition="top center")

    # Quadrant lines at median values
    med_x = sub["Rigid Score"].median()
    med_y = sub["Coverage"].median()
    fig.add_hline(y=med_y, line_dash="dash", line_color="white", opacity=0.3)
    fig.add_vline(x=med_x, line_dash="dash", line_color="white", opacity=0.3)

    # Quadrant labels
    for label, x, y, anchor in [
        ("True Hit",    sub["Rigid Score"].min() * 1.05, 0.95, "right"),
        ("False Hit",   sub["Rigid Score"].max() * 0.95, 0.95, "left"),
        ("True Miss",   sub["Rigid Score"].min() * 1.05, 0.05, "right"),
        ("False Miss",  sub["Rigid Score"].max() * 0.95, 0.05, "left"),
    ]:
        fig.add_annotation(x=x, y=y, text=label, showarrow=False,
                           font=dict(color="gray", size=11), xanchor=anchor)

    fig.update_layout(template="plotly_dark", height=600)
    fig.write_html(f"scatter_{tname}.html")
    fig.show()

## Download Results

In [ ]:
import zipfile, os
from IPython.display import FileLink, display

files = (
    ["spade_benchmark_results.csv"]
    + [f"scatter_{t}.html" for t in TARGETS]
)

with zipfile.ZipFile("spade_benchmark.zip", "w") as zf:
    for f in files:
        if os.path.exists(f):
            zf.write(f)

print("Packaged:", files)
display(FileLink("spade_benchmark.zip"))